In [1]:
import warnings
warnings.filterwarnings("ignore", message=".*pandas only supports SQLAlchemy.*")
warnings.filterwarnings("ignore", category=RuntimeWarning)

import pymysql
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import ipywidgets as widgets
from IPython.display import display, clear_output

# 设置绘图风格与中文字体
plt.rcParams['font.sans-serif'] = ['SimHei'] 
plt.rcParams['axes.unicode_minus'] = False
sns.set_theme(style="white", font='SimHei')

def get_db_connection():
    """建立 StarRocks 数据库连接"""
    return pymysql.connect(
        host='192.168.144.101',
        port=9030,
        user='hadoop',
        password='hadoop',
        charset='utf8'
    )

def fetch_cities():
    """获取 DWD 表中有数据的城市列表"""
    conn = get_db_connection()
    query = """
    SELECT DISTINCT city 
    FROM iceberg_catalog.ershoufang.dwd_ershoufang_fact 
    WHERE city IS NOT NULL AND city != ''
    ORDER BY city
    """
    try:
        df = pd.read_sql(query, conn)
        return df['city'].tolist()
    finally:
        conn.close()

def fetch_price_data(city_name):
    """
    获取指定城市的总价和单价数据。
    用于计算总价分箱和各箱内的平均单价。
    """
    conn = get_db_connection()
    query = f"""
    SELECT 
        total_price,
        unit_price
    FROM iceberg_catalog.ershoufang.dwd_ershoufang_fact 
    WHERE city = '{city_name}' 
      AND total_price IS NOT NULL 
      AND unit_price IS NOT NULL
      AND unit_price > 0
    """
    try:
        df = pd.read_sql(query, conn)
        return df
    finally:
        conn.close()

In [ ]:
def plot_dual_axis_chart(city):
    """绘制双 Y 轴组合图 (柱状图+折线图)"""
    df = fetch_price_data(city)
    
    if df.empty or len(df) < 50:
        plt.figure(figsize=(10, 6))
        plt.text(0.5, 0.5, f"{city} 有效数据不足，无法生成趋势图", ha='center', va='center', fontsize=14)
        plt.axis('off')
        plt.show()
        return

    # --- 1. 自适应价格区间划分 (动态分箱) ---
    quantiles = df['total_price'].quantile([0.2, 0.4, 0.6, 0.8]).tolist()
    bins = [-1] + [q + (i * 0.0001) for i, q in enumerate(quantiles)] + [float('inf')]
    
    labels = [
        f'低价位\n(<{int(quantiles[0])}万)', 
        f'中低价\n({int(quantiles[0])}-{int(quantiles[1])}万)', 
        f'中等价\n({int(quantiles[1])}-{int(quantiles[2])}万)', 
        f'中高价\n({int(quantiles[2])}-{int(quantiles[3])}万)', 
        f'高价位\n(>{int(quantiles[3])}万)'
    ]
    df['price_tier'] = pd.cut(df['total_price'], bins=bins, labels=labels)

    # --- 2. 数据聚合 ---
    stats = df.groupby('price_tier').agg(
        house_count=('unit_price', 'size'),
        avg_unit_price=('unit_price', 'mean')
    ).reset_index()

    if stats.empty:
        return

    # --- 3. 开始绘图 ---
    fig, ax1 = plt.subplots(figsize=(12, 6))
    x_pos = np.arange(len(stats['price_tier']))

    # 【主 Y 轴 (左侧)】：房源数量 (柱状图)
    color_bar = '#BDC3C7'
    ax1.bar(x_pos, stats['house_count'], color=color_bar, alpha=0.5, width=0.5, label='房源数量 (左轴)')
    ax1.set_ylabel('房源数量 (套)', fontsize=12, color='#7F8C8D')
    ax1.tick_params(axis='y', labelcolor='#7F8C8D')
    ax1.set_xticks(x_pos)
    ax1.set_xticklabels(stats['price_tier'], fontsize=11)

    # 【次 Y 轴 (右侧)】：平均单价 (折线图)
    ax2 = ax1.twinx()  
    color_line = '#E74C3C' 
    ax2.plot(x_pos, stats['avg_unit_price'], color=color_line, marker='o', 
             linewidth=3, markersize=8, markerfacecolor='white', markeredgewidth=2, label='平均单价 (右轴)')
    ax2.set_ylabel('平均单价 (元/平米)', fontsize=12, color=color_line)
    ax2.tick_params(axis='y', labelcolor=color_line)

    # 在折线节点上方添加具体的单价数值
    for i, txt in enumerate(stats['avg_unit_price']):
        ax2.text(x_pos[i], txt + (stats['avg_unit_price'].max() * 0.02), 
                 f"{int(txt):,}", ha='center', va='bottom', fontsize=11, fontweight='bold', color=color_line)

    # --- 4. 图表修饰 ---
    plt.title(f'[{city}] 价格区间单价爬坡趋势 (均价折线 + 房源量背景)', fontsize=16, pad=20)
    
    # 巧妙合并两个轴的图例
    lines_1, labels_1 = ax1.get_legend_handles_labels()
    lines_2, labels_2 = ax2.get_legend_handles_labels()
    ax2.legend(lines_1 + lines_2, labels_1 + labels_2, loc='upper left', frameon=True, shadow=True)

    # 移除顶部边框，保留左右Y轴线
    ax1.spines['top'].set_visible(False)
    ax2.spines['top'].set_visible(False)
    ax1.grid(axis='y', linestyle=':', alpha=0.4)

    plt.tight_layout()
    plt.show()

In [3]:
def render_interactive_dashboard():
    """构建交互式下拉组件"""
    cities = fetch_cities()
    if not cities:
        return
    
    default_city = '北京' if '北京' in cities else cities[0]
    
    city_dropdown = widgets.Dropdown(
        options=cities,
        value=default_city,
        description='📍 分析城市:',
        layout={'width': '250px'}
    )
    
    plot_output = widgets.Output()

    def on_city_change(change):
        if change['type'] == 'change' and change['name'] == 'value':
            with plot_output:
                clear_output(wait=True)
                plot_dual_axis_chart(change['new'])

    city_dropdown.observe(on_city_change)

    display(city_dropdown)
    display(plot_output)
    
    with plot_output:
        plot_dual_axis_chart(default_city)

In [4]:
if __name__ == "__main__":
    render_interactive_dashboard()

Dropdown(description='📍 分析城市:', index=4, layout=Layout(width='250px'), options=('上海', '东莞', '中山', '佛山', '北京', …

Output()